# IS 362 — Week 8: Auto MPG Exploratory Data Analysis

**Goal:** Load the UCI Auto MPG dataset directly from the web, clean it reproducibly, and explore relationships with charts.

## 1–3. Load the dataset and name columns
The [UCI Auto MPG dataset](https://archive.ics.uci.edu/dataset/9/auto+mpg) describes fuel efficiency and vehicle characteristics. The dataset documentation lists eight attributes: mpg, cylinders, displacement, horsepower, weight, acceleration, model year, and origin. The raw file also includes a car-name field. The raw dataset has 398 rows. Origin is encoded as 1=USA, 2=Europe, 3=Japan in the original documentation; this assignment specifically requests 1=USA, 2=Asia, 3=Europe, so the assignment mapping is used below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
columns = ["mpg", "cylinders", "displacement", "horsepower", "weight",
           "acceleration", "model_year", "origin", "car_name"]
df = pd.read_csv(url, sep=r"\s+", names=columns, na_values="?", quotechar=""", engine="python")
print("Dataset shape:", df.shape)
display(df.head())

## 4–5. Clean horsepower and transform origin
The `?` placeholders are interpreted as missing values on import. We explicitly convert horsepower to numeric, then apply the origin labels **as specified in the assignment**. Note that this differs from the UCI documentation, which identifies 2 as Europe and 3 as Japan.

In [ ]:
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
df["origin"] = df["origin"].map({1: "USA", 2: "Asia", 3: "Europe"})
print(df.info())
print("\nMissing values by column:\n", df.isna().sum())
display(df.describe(include="all"))

## 6. Bar chart: Distribution of cylinders
This chart counts the number of cars with each cylinder count.

In [ ]:
counts = df["cylinders"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(counts.index.astype(str), counts.values, color="steelblue")
ax.set(title="Distribution of Engine Cylinders", xlabel="Number of Cylinders", ylabel="Number of Cars")
for i, value in enumerate(counts.values):
    ax.text(i, value + 2, str(value), ha="center")
plt.tight_layout()
plt.show()
print("Most common cylinder count:", counts.idxmax())

## 7. Scatterplot: Horsepower vs. weight
Each point represents a vehicle with recorded horsepower. Missing horsepower values are excluded only for this plot.

In [ ]:
hp_weight = df.dropna(subset=["horsepower", "weight"])
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=hp_weight, x="horsepower", y="weight", alpha=0.65, ax=ax)
ax.set(title="Horsepower vs. Vehicle Weight", xlabel="Horsepower", ylabel="Weight (lbs)")
plt.tight_layout()
plt.show()
print("Pearson correlation:", round(hp_weight["horsepower"].corr(hp_weight["weight"]), 3))

## 8. Research question: Do cars with more horsepower tend to have lower MPG?
We use a scatterplot with a fitted trend line, plus a correlation coefficient, to investigate whether higher horsepower is associated with lower fuel economy. Correlation does not prove causation.

In [ ]:
analysis = df.dropna(subset=["horsepower", "mpg"])
fig, ax = plt.subplots(figsize=(9, 5))
sns.regplot(data=analysis, x="horsepower", y="mpg",
            scatter_kws={"alpha": 0.45}, line_kws={"color": "crimson"}, ax=ax)
ax.set(title="Horsepower vs. Fuel Efficiency", xlabel="Horsepower", ylabel="Miles Per Gallon (MPG)")
plt.tight_layout()
plt.show()
r = analysis["horsepower"].corr(analysis["mpg"])
print(f"Pearson correlation (horsepower vs MPG): {r:.3f}")
print("Answer: The relationship is negative: cars with greater horsepower generally have lower MPG." if r < 0 else "Answer: This dataset does not show a negative linear relationship.")

## Conclusions
- The cylinder bar chart identifies which engine size categories are most common.
- The horsepower–weight scatterplot shows whether heavier cars generally have more horsepower.
- The horsepower–MPG scatterplot and correlation test whether more powerful cars tend to have lower fuel economy.
- Six records have missing horsepower in the original dataset; retaining them in the main DataFrame avoids discarding information unnecessarily.

**Reproducibility:** Run all cells from top to bottom in Jupyter. The notebook downloads the source file directly, and every cleanup operation is included in code. Requires `pandas`, `numpy`, `matplotlib`, and `seaborn`.

**Source:** [UCI Machine Learning Repository — Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg).